# Matrix multiplication, softmax, attention math

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Matrix multiplication

> **Problem.** A model has to apply the same transformation to every token of every sequence in a batch — millions of times per second. Doing it token by token would take hours; the whole point of a GPU is that it does this as one operation.

**Idea.** Multiply a matrix of inputs by a matrix of weights: every row of inputs is transformed at once.

**Use when** every neural network layer, attention score, embedding projection.  
**Not when** —.

```
tokens (4 × 8)   @   weights (8 × 3)   =   output (4 × 3)
  4 tokens,           8 in → 3 out           4 tokens,
  8 features each                            3 features each
inner dimensions must match: 8 = 8
```

**How it works.**
1. `tokens @ weights` treats each of the 4 rows as a vector and computes its dot product with each of the 3 weight columns — 12 dot products in one call.
2. The inner dimensions must agree (8 and 8); `tokens @ tokens` fails because 8 ≠ 4.
3. A batch adds a leading dimension: `(16, 4, 8) @ (8, 3)` applies the same weights to 16 sequences at once.
4. This is the operation GPUs are built for; a transformer forward pass is thousands of these.

| | what happens | result |
|:--|:--|:--|
| ✗ loop | one token at a time | 4 small operations, slow on GPU |
| ✓ matmul | `tokens @ weights` | one operation, hardware-optimised |

**Production code and its real output**

In [2]:
# Matrix multiplication — every neural layer is one. torch is the production tool.
import torch

torch.manual_seed(0)
tokens = torch.randn(4, 8)  # 4 tokens, 8 features each
weights = torch.randn(8, 3)  # a layer projecting 8 features → 3

projected = tokens @ weights
print("tokens (4×8) @ weights (8×3) → ", tuple(projected.shape))

# Shapes must agree on the inner dimension, or torch refuses.
try:
    tokens @ tokens
except RuntimeError as error:
    print("tokens @ tokens →", str(error)[:60])

# Batched: 16 sequences at once, one call.
batch = torch.randn(16, 4, 8)
print("batch (16×4×8) @ weights →", tuple((batch @ weights).shape))
assert tuple(projected.shape) == (4, 3)

tokens (4×8) @ weights (8×3) →  (4, 3)
tokens @ tokens → mat1 and mat2 shapes cannot be multiplied (4x8 and 4x8)
batch (16×4×8) @ weights → (16, 4, 3)


**What the output shows.** (4×8) @ (8×3) produced a (4×3) output, the mismatched shapes raised an error naming the sizes, and the batched product handled 16 sequences in one call.

**In practice**
- **shapes are the bug** — most model bugs are shape bugs; print `.shape` at every boundary and use `einops` for readability.
- **batch everything** — a loop calling matmul per item throws away the GPU; stack inputs into a batch dimension.
- **precision** — matmul in bf16/fp16 is 2–8× faster on GPUs with negligible loss for inference (layer 1.2).
- **memory** — activations of shape (batch, seq, features) are what fill GPU memory, not the weights — watch the product of those three numbers.

**Alternatives** — `torch.einsum` for readable multi-axis products · `np.matmul` on CPU-only paths

**Terms** — *matmul*: matrix multiplication, `@` in Python · *batch dimension*: the leading axis that indexes independent examples


### softmax

> **Problem.** A model ends with one raw score per possible next token — 100,000 numbers, some negative, some in the thousands. To pick a token, or to say how confident it is, those scores must become probabilities that add up to 1.

**Idea.** Exponentiate each score and divide by the sum: bigger scores get bigger shares, everything sums to 1.

**Use when** turning scores into probabilities: next-token choice, attention weights, classification.  
**Not when** you only need the top score — `argmax` is cheaper and gives the same winner.

```
logits   [ 2.0   1.0   0.1 ]
   exp   [ 7.39  2.72  1.11 ]   sum = 11.2
softmax  [ 0.66  0.24  0.10 ]   sums to 1

÷ temperature first:  T=0.5 → [0.87 0.12 0.02]   T=2 → [0.48 0.29 0.23]
```

**How it works.**
1. `torch.softmax(logits, dim=0)` exponentiates each score and divides by the total; the output sums to 1.
2. Dividing the logits by a temperature before softmax sharpens (T<1) or flattens (T>1) the distribution — that is what the `temperature` API parameter does.
3. torch subtracts the maximum internally before exponentiating, so `[1000, 999]` gives `[0.73, 0.27]` instead of overflowing to `nan`.
4. Attention uses the same function across each row of scores so every token's weights sum to 1.

| | what happens | result |
|:--|:--|:--|
| ✗ naive exp/sum | `exp(1000)` | overflow → nan |
| ✓ torch.softmax | same logits | `[0.731, 0.269]` |
| ✓ T=0.5 | sharper | top token 0.87 |
| ✓ T=2 | flatter | top token 0.48 |

**Production code and its real output**

In [3]:
# Softmax — scores → probabilities that sum to 1. Use torch.softmax; never write exp/sum by hand.
import torch

logits = torch.tensor([2.0, 1.0, 0.1])
probabilities = torch.softmax(logits, dim=0)
print("logits:      ", logits.tolist())
print("softmax:     ", probabilities.numpy(), "sum =", round(float(probabilities.sum()), 6))

# Temperature: divide logits before softmax. Low → peaked, high → flat.
for temperature in [0.5, 1.0, 2.0]:
    print(f"temperature {temperature}:", torch.softmax(logits / temperature, dim=0).numpy())

# Large logits: torch's implementation is numerically stable; a naive exp() overflows.
big = torch.tensor([1000.0, 999.0])
print("softmax([1000, 999]):", torch.softmax(big, dim=0).numpy())
assert (
    abs(float(probabilities.sum()) - 1) < 1e-6 and not torch.isnan(torch.softmax(big, dim=0)).any()
)

logits:       [2.0, 1.0, 0.10000000149011612]
softmax:      [0.659  0.2424 0.0986] sum = 1.0
temperature 0.5: [0.8638 0.1169 0.0193]
temperature 1.0: [0.659  0.2424 0.0986]
temperature 2.0: [0.5017 0.3043 0.194 ]
softmax([1000, 999]): [0.7311 0.2689]


**What the output shows.** The three logits became probabilities summing to 1; lowering temperature pushed mass onto the top token, raising it spread mass out; the 1000/999 case stayed finite.

**In practice**
- **never hand-roll** — `exp` overflows above ~88 in float32; every library subtracts the max — use the library.
- **log-softmax for loss** — cross-entropy uses `log_softmax` directly for numerical stability; do not `log(softmax(x))`.
- **temperature is not free** — T=0 makes outputs deterministic but not necessarily better; sampling parameters are in the next notebook.
- **dim matters** — softmax over the wrong axis silently produces nonsense that still sums to 1 — always pass `dim` explicitly.

**Alternatives** — `argmax` when only the winner matters · sigmoid for independent yes/no scores

**Terms** — *logit*: a raw, unnormalised score · *temperature*: a divisor applied to logits before softmax · *distribution*: probabilities that sum to 1


### attention math

> **Problem.** In "the bank raised rates because it feared inflation", the word *it* means *the bank*. A model reading one token at a time needs a way for *it* to look back at every earlier token and decide which ones matter.

**Idea.** Each token asks a question (Q) of every other token's key (K); the softmaxed scores weight the values (V) it gathers.

**Use when** understanding or debugging any transformer; implementing a custom attention variant.  
**Not when** writing model code — call `scaled_dot_product_attention`, do not reimplement it.

```mermaid
flowchart LR
    Q["Q · Kᵀ"] --> S["÷ √d"] --> M["causal mask · no looking ahead"] --> SM["softmax · rows sum to 1"] --> V["× V"] --> O[output]
```

**How it works.**
1. Q, K and V are three projections of the same token vectors: shape (batch, heads, seq, d_head).
2. `Q @ Kᵀ / √d` gives a seq × seq score matrix: how much each token attends to each other token; √d keeps the scores from growing with dimension.
3. The causal mask sets scores for future tokens to −∞ so a decoder cannot see ahead; softmax over each row turns the rest into weights that sum to 1.
4. `weights @ V` mixes the value vectors by those weights — each token's output is a weighted summary of what it attended to.
5. `F.scaled_dot_product_attention(q, k, v, is_causal=True)` does all of this in one fused kernel; the cell confirms the written-out version matches it.

| | what happens | result |
|:--|:--|:--|
| ✗ hand-written | materialises the seq × seq matrix | O(seq²) memory, slow |
| ✓ SDPA kernel | same math, fused | FlashAttention on GPU: exact, linear memory |

**Production code and its real output**

In [4]:
# Attention math — softmax(Q Kᵀ / √d) V. Production code calls F.scaled_dot_product_attention,
# which picks a fused kernel (FlashAttention on GPUs).
import torch
import torch.nn.functional as F

torch.manual_seed(0)
batch, heads, seq, d_head = 1, 1, 3, 4
q = torch.randn(batch, heads, seq, d_head)
k = torch.randn(batch, heads, seq, d_head)
v = torch.randn(batch, heads, seq, d_head)

output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("output shape:", tuple(output.shape), "= (batch, heads, seq, d_head)")

# The same numbers, written out once to see what the kernel computes.
scores = q @ k.transpose(-2, -1) / d_head**0.5
mask = torch.triu(torch.ones(seq, seq, dtype=torch.bool), diagonal=1)  # causal: no looking ahead
weights = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
print("attention weights (rows sum to 1, upper triangle is 0):")
print(weights[0, 0].numpy())
print("matches the fused kernel:", torch.allclose(weights @ v, output, atol=1e-5))
assert torch.allclose(weights @ v, output, atol=1e-5)

output shape: (1, 1, 3, 4) = (batch, heads, seq, d_head)
attention weights (rows sum to 1, upper triangle is 0):
[[1.     0.     0.    ]
 [0.5735 0.4265 0.    ]
 [0.4543 0.3873 0.1584]]
matches the fused kernel: True


**What the output shows.** The fused kernel and the written-out math agree to 1e-5; the printed weight matrix has rows summing to 1 and zeros above the diagonal (the causal mask).

**In practice**
- **cost is quadratic** — the score matrix is seq × seq per head; doubling context quadruples attention memory — the reason for FlashAttention and sliding windows (layer 1.1).
- **always use SDPA** — it selects FlashAttention or memory-efficient kernels automatically; hand-written attention is 5–20× slower on GPU.
- **the mask is the model type** — causal mask = decoder (GPT); no mask = encoder (BERT); both is encoder-decoder (T5).
- **multi-head is just batching** — heads are independent attention computations run in parallel and concatenated (layer 1.1).

**Alternatives** — linear/sparse attention variants for very long context · state-space models (Mamba) with no quadratic term

**Terms** — *Q / K / V*: query, key, value — three views of each token · *causal mask*: hide future tokens · *head*: one independent attention computation
